# Mammalian Dataset Pipeline Walkthrough

This notebook walks through every step of the OMA data download pipeline
for building a balanced mammalian protein dataset. Each cell makes real API
calls (on small samples) so you can inspect the data at every stage.

## Pipeline overview

```
Phase 1: GET /api/genome/{code}/           → species metadata (30 calls)
Phase 2: GET /api/genome/{code}/proteins/  → protein inventories (~4500 paginated calls)
Phase 3: Local HOG-balanced sampling       → sampled protein IDs (no API)
Phase 4: Fetch sequences (bulk or individual) → FASTA files
Phase 5: Local merge                       → final mammalia_dataset.feather
```

**Important:** This notebook uses MOUSE and HUMAN as test species. The full
pipeline will run on all 30 species via a script.

In [2]:
import requests
import time
import json
import gzip
import numpy as np
import pandas as pd
from pathlib import Path
from io import StringIO
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

OMA_API = "https://omabrowser.org/api"

# All 30 selected species across 14 mammalian orders
SPECIES_CODES = [
    "ORNAN", "TACAU",  # Monotremata (platypus, echidna)
    "MONDO", "SARHA",  # Metatheria (opossum, Tasmanian devil)
    "LOXAF", "ECHTE",  # Afrotheria (elephant, tenrec)
    "DASNO",            # Xenarthra (armadillo)
    "ERIEU",            # Eulipotyphla (hedgehog)
    "RHIFE", "MYOLU",  # Chiroptera (horseshoe bat, little brown bat)
    "FELCA", "CANLF", "AILME",  # Carnivora (cat, dog, panda)
    "HORSE",            # Perissodactyla (horse)
    "BOVIN", "PIGXX", "TURTR",  # Artiodactyla (cow, pig, dolphin)
    "MANJA",            # Pholidota (pangolin)
    "HUMAN", "MACMU", "CALJA", "NOMLE", "MICMU",  # Primates
    "MOUSE", "RATNO", "CAVPO", "HETGA",  # Rodentia
    "RABIT",            # Lagomorpha (rabbit)
    "TUPBE",            # Scandentia (tree shrew)
    "BALMU",            # Artiodactyla/Cetacea (blue whale)
]

# Test with just 2 species for this walkthrough
TEST_SPECIES = ["MOUSE", "HUMAN"]

print(f"Full pipeline: {len(SPECIES_CODES)} species")
print(f"This walkthrough: {TEST_SPECIES}")

Full pipeline: 30 species
This walkthrough: ['MOUSE', 'HUMAN']


---
## Phase 1: Fetch Species Metadata

**Endpoint:** `GET /api/genome/{code}/`

For each species code (e.g. `MOUSE`), this returns:
- `species` — scientific name
- `taxon_id` — NCBI taxonomy ID
- `nr_entries` — total number of proteins in OMA for this genome
- `lineage` — full taxonomic lineage (list of strings)

We need this to:
1. Know how many proteins each species has (for setting the sampling target)
2. Store species metadata alongside the dataset for provenance

In [4]:
# --- Phase 1: GET /api/genome/{code}/ ---
# One call per species. Returns metadata including protein count.

species_meta = {}
for code in TEST_SPECIES:
    url = f"{OMA_API}/genome/{code}/"
    print(f"Fetching metadata for {code}...")
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    data = r.json()
    
    species_meta[code] = {
        "code": code,
        "species": data["species"],
        "taxon_id": data["taxon_id"],
        "nr_entries": data["nr_entries"],
        # First few lineage entries (most specific → least specific)
        "lineage_sample": data["lineage"],
    }
    time.sleep(0.5)  # polite rate limiting

# Display what we got
for code, meta in species_meta.items():
    print(f"\n{code}:")
    print(f"  Species:    {meta['species']}")
    print(f"  Taxon ID:   {meta['taxon_id']}")
    print(f"  # Proteins: {meta['nr_entries']:,}")
    print(f"  Lineage:    {' > '.join(meta['lineage_sample'])}")

Fetching metadata for MOUSE...
Fetching metadata for HUMAN...

MOUSE:
  Species:    Mus musculus
  Taxon ID:   10090
  # Proteins: 67,158
  Lineage:    Mus musculus > Mus > Murinae > Muridae > Muroidea > Myomorpha > Rodentia > Glires > Euarchontoglires > Boreoeutheria > Eutheria > Theria > Mammalia > Amniota > Tetrapoda > Dipnotetrapodomorpha > Sarcopterygii > Euteleostomi > Teleostomi > Gnathostomata > Vertebrata > Craniata > Chordata > Deuterostomia > Bilateria > Eumetazoa > Metazoa > Opisthokonta > Eukaryota

HUMAN:
  Species:    Homo sapiens
  Taxon ID:   9606
  # Proteins: 103,200
  Lineage:    Homo sapiens > Homo > Homininae > Hominidae > Hominoidea > Catarrhini > Simiiformes > Haplorrhini > Primates > Euarchontoglires > Boreoeutheria > Eutheria > Theria > Mammalia > Amniota > Tetrapoda > Dipnotetrapodomorpha > Sarcopterygii > Euteleostomi > Teleostomi > Gnathostomata > Vertebrata > Craniata > Chordata > Deuterostomia > Bilateria > Eumetazoa > Metazoa > Opisthokonta > Eukaryota


### What the raw API response looks like

Let's inspect the full response for one species so you can see all available fields.

In [5]:
# Raw response for MOUSE (truncating large fields)
r = requests.get(f"{OMA_API}/genome/MOUSE/", timeout=30)
raw = r.json()
# Show keys and non-list values
for k, v in raw.items():
    if isinstance(v, list):
        print(f"  {k}: [{len(v)} items] e.g. {v[0] if v else '(empty)'}...")
    elif isinstance(v, str) and len(v) > 80:
        print(f"  {k}: {v[:80]}...")
    else:
        print(f"  {k}: {v}")

  code: MOUSE
  taxon_id: 10090
  species: Mus musculus
  nr_entries: 67158
  lineage: [29 items] e.g. Mus musculus...
  proteins: https://omabrowser.org/api/genome/MOUSE/proteins/
  chromosomes: [35 items] e.g. {'id': '10', 'entry_ranges': [[12630472, 12633558]]}...


---
## Phase 2: Fetch Protein Lists (no sequences)

**Endpoint:** `GET /api/genome/{code}/proteins/?per_page=500`

This is the workhorse call. For each species, we paginate through ALL proteins
and collect:
- `omaid` — unique OMA protein ID (e.g. `MOUSE00001`)
- `oma_hog_id` — full HOG path (e.g. `HOG:E0761743.3b.1b`)
- `sequence_length` — amino acid count
- `canonicalid` — cross-reference ID (often UniProt)

**Key detail:** This endpoint does NOT return sequences. That's intentional —
it makes the response small and fast. Sequences come later in Phase 4.

**Pagination:** The API returns:
- `x-total-count` header: total proteins for this species
- `Link` header: URL for the next page

For MOUSE (67K proteins), this takes ~135 pages × 1 req/sec ≈ 2 minutes.
For all 30 species: ~50 minutes total.

In [6]:
# --- Phase 2: Fetch protein list (demo: first 3 pages of MOUSE) ---

def fetch_protein_page(species_code: str, page: int = 1, per_page: int = 500):
    """Fetch one page of proteins for a species. Returns (records, total_count, next_url)."""
    url = f"{OMA_API}/genome/{species_code}/proteins/"
    r = requests.get(url, params={"per_page": per_page, "page": page}, timeout=30)
    r.raise_for_status()
    
    total = int(r.headers.get("x-total-count", 0))
    
    # Parse Link header for next page URL
    link_header = r.headers.get("Link", "")
    next_url = None
    for part in link_header.split(","):
        if 'rel="next"' in part:
            next_url = part.split(";")[0].strip().strip("<>")
    
    return r.json(), total, next_url


# Fetch first 3 pages to examine the data
all_records = []
next_url = None
for page in range(1, 4):
    records, total, next_url = fetch_protein_page("MOUSE", page=page)
    all_records.extend(records)
    print(f"Page {page}: got {len(records)} proteins (total: {total:,}, next: {'yes' if next_url else 'no'})")
    time.sleep(0.5)

print(f"\nFetched {len(all_records)} proteins from 3 pages")
print(f"Total available for MOUSE: {total:,}")
print(f"Pages needed at 500/page: {(total + 499) // 500}")

Page 1: got 500 proteins (total: 67,158, next: yes)
Page 2: got 500 proteins (total: 67,158, next: yes)
Page 3: got 500 proteins (total: 67,158, next: yes)

Fetched 1500 proteins from 3 pages
Total available for MOUSE: 67,158
Pages needed at 500/page: 135


### Inspect the protein list data

Let's look at what fields each protein record contains and how the HOG IDs look.

In [7]:
# Convert to DataFrame and inspect
df = pd.DataFrame(all_records)
print("Columns returned by the protein list endpoint:")
print(f"  {list(df.columns)}")
print(f"\nShape: {df.shape}")
print()

# Show a few rows — these are the fields we keep
display(df[["omaid", "canonicalid", "oma_hog_id", "sequence_length"]].head(10))

# HOG ID breakdown
print("\nHOG ID examples:")
for _, row in df.head(5).iterrows():
    hog = row["oma_hog_id"]
    if hog:
        parts = hog.split(".")
        root = parts[0]  # e.g. "HOG:E0761743"
        depth = len(parts) - 1
        print(f"  {row['omaid']}: {hog}  →  root={root}, depth={depth}")
    else:
        print(f"  {row['omaid']}: (no HOG assigned)")

# How many proteins have a HOG assignment?
n_with_hog = df["oma_hog_id"].notna().sum()
n_without = df["oma_hog_id"].isna().sum()
print(f"\nIn this sample: {n_with_hog} with HOG, {n_without} without ({100*n_without/len(df):.1f}% orphan)")

Columns returned by the protein list endpoint:
  ['entry_nr', 'entry_url', 'omaid', 'canonicalid', 'sequence_md5', 'sequence_length', 'species', 'oma_group', 'oma_hog_id', 'chromosome', 'locus', 'is_main_isoform']

Shape: (1500, 12)



,omaid,canonicalid,oma_hog_id,sequence_length
0,MOUSE00001,NK2R_MOUSE,HOG:E0761743.3b.1b,384
1,MOUSE00002,OCC1_MOUSE,HOG:E0740156.1a,63
2,MOUSE00003,A0A1W2P6X0,,61
3,MOUSE00004,H3BJL1,,190
4,MOUSE00005,RHG18_MOUSE,HOG:E0770201.2b.1a.3b.2a,663
5,MOUSE00006,H3BLG9,,73
6,MOUSE00007,NUAK1_MOUSE,HOG:E0754428.1a,658
7,MOUSE00008,F6XZX4,,160
8,MOUSE00009,CKAP4_MOUSE,HOG:E0746856.2a.6a,575
9,MOUSE00010,CKAP4_MOUSE,,575



HOG ID examples:
  MOUSE00001: HOG:E0761743.3b.1b  →  root=HOG:E0761743, depth=2
  MOUSE00002: HOG:E0740156.1a  →  root=HOG:E0740156, depth=1
  MOUSE00003: (no HOG assigned)
  MOUSE00004: (no HOG assigned)
  MOUSE00005: HOG:E0770201.2b.1a.3b.2a  →  root=HOG:E0770201, depth=4

In this sample: 1500 with HOG, 0 without (0.0% orphan)


---
## Phase 3: HOG-Balanced Sampling

This phase is **pure local computation** — no API calls.

Given a protein list with HOG assignments, the sampling algorithm:

1. **Filter** to proteins that have a HOG assignment (drop orphans)
2. **Extract root HOG** from the full HOG path (`HOG:E0761743.3b.1b` → `HOG:E0761743`)
3. **Round 1 — diversity pass:** Pick exactly 1 random protein from each unique root HOG.
   This maximizes the number of distinct gene families represented.
4. **Round 2 — fill pass:** If we still need more proteins to hit the target,
   go back to the largest HOGs (most paralogs) and sample additional members.

The result: every gene family appears at least once, and large families contribute
proportionally more — but no family dominates the sample.

In [8]:
# --- Phase 3: HOG-balanced sampling on our demo data ---

def hog_balanced_sample(protein_df: pd.DataFrame, target_n: int, seed: int = 42) -> pd.DataFrame:
    """
    Sample proteins with maximum HOG diversity.
    
    Args:
        protein_df: DataFrame with columns [omaid, oma_hog_id, ...]
        target_n: desired number of proteins
        seed: random seed for reproducibility
    
    Returns:
        DataFrame of sampled proteins with added 'roothog_id' column
    """
    rng = np.random.default_rng(seed)
    
    # Step 1: Filter to proteins with HOG assignments
    has_hog = protein_df[protein_df["oma_hog_id"].notna()].copy()
    print(f"Proteins with HOG: {len(has_hog)} / {len(protein_df)} "
          f"({100*len(has_hog)/len(protein_df):.1f}%)")
    
    # Step 2: Extract root HOG (everything before the first '.')
    has_hog["roothog_id"] = has_hog["oma_hog_id"].str.split(".").str[0]
    
    # Step 3: Round 1 — diversity pass (1 protein per root HOG)
    round1 = has_hog.groupby("roothog_id").apply(
        lambda g: g.sample(1, random_state=rng.integers(1e9))
    ).reset_index(drop=True)
    print(f"Round 1 (diversity): {len(round1)} proteins from {round1['roothog_id'].nunique()} unique HOGs")
    
    if len(round1) >= target_n:
        # More HOGs than target — subsample the diversity set
        sampled = round1.sample(target_n, random_state=rng.integers(1e9))
        print(f"Target {target_n} ≤ unique HOGs {len(round1)}, subsampled diversity set")
        return sampled
    
    # Step 4: Round 2 — fill pass from largest HOGs
    remaining_target = target_n - len(round1)
    already_sampled = set(round1["omaid"])
    pool = has_hog[~has_hog["omaid"].isin(already_sampled)]
    
    # Sort HOGs by size (largest first) to fill proportionally
    hog_sizes = pool.groupby("roothog_id").size().sort_values(ascending=False)
    print(f"Round 2 pool: {len(pool)} remaining proteins across {len(hog_sizes)} HOGs")
    print(f"Need {remaining_target} more proteins")
    
    # Sample proportionally from largest HOGs
    fill_records = []
    for hog_id in hog_sizes.index:
        if remaining_target <= 0:
            break
        hog_proteins = pool[pool["roothog_id"] == hog_id]
        n_take = min(len(hog_proteins), max(1, remaining_target * len(hog_proteins) // len(pool)))
        n_take = min(n_take, remaining_target)  # don't overshoot
        if n_take > 0:
            fill_records.append(hog_proteins.sample(n_take, random_state=rng.integers(1e9)))
            remaining_target -= n_take
    
    if fill_records:
        round2 = pd.concat(fill_records)
        print(f"Round 2 (fill): {len(round2)} additional proteins")
        sampled = pd.concat([round1, round2])
    else:
        sampled = round1
    
    print(f"\nFinal sample: {len(sampled)} proteins, {sampled['roothog_id'].nunique()} unique HOGs")
    return sampled


# Run sampling on our 1500-protein demo set
TARGET_N = 500  # small target for demo
sampled = hog_balanced_sample(df, target_n=TARGET_N)
print(f"\n--- Sampling result ---")
print(f"Sampled: {len(sampled)} proteins")
print(f"Unique root HOGs: {sampled['roothog_id'].nunique()}")
print(f"Target was: {TARGET_N}")

Proteins with HOG: 1500 / 1500 (100.0%)
Round 1 (diversity): 424 proteins from 424 unique HOGs
Round 2 pool: 1076 remaining proteins across 22 HOGs
Need 76 more proteins
Round 2 (fill): 76 additional proteins

Final sample: 500 proteins, 424 unique HOGs

--- Sampling result ---
Sampled: 500 proteins
Unique root HOGs: 424
Target was: 500


/tmp/ipykernel_320644/2414790545.py:26: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  round1 = has_hog.groupby("roothog_id").apply(


In [11]:
sampled

,entry_nr,entry_url,omaid,canonicalid,sequence_md5,sequence_length,species,oma_group,oma_hog_id,chromosome,locus,is_main_isoform,roothog_id
0,12630767,https://omabrowser.org/api/protein/12630767/,MOUSE00296,A0A1W2P7U8,6e1e43a6eefc37e94397147ca8f44f21,245,"{'code': 'MOUSE', 'taxon_id': 10090, 'species'...",0,,10,"{'start': 33468843, 'end': 33498928, 'strand':...",False,
1,12631409,https://omabrowser.org/api/protein/12631409/,MOUSE00938,LRB4A_MOUSE,5e300b066d6e3920d06d5a68e6895166,335,"{'code': 'MOUSE', 'taxon_id': 10090, 'species'...",1183802,HOG:E0718947.1a,10,"{'start': 51367114, 'end': 51372340, 'strand': 1}",True,HOG:E0718947
2,12631938,https://omabrowser.org/api/protein/12631938/,MOUSE01467,GRPL2_MOUSE,19059f3558585d2056fab8a28c10223c,332,"{'code': 'MOUSE', 'taxon_id': 10090, 'species'...",1076419,HOG:E0726706,10,"{'start': 111919281, 'end': 111943145, 'strand...",True,HOG:E0726706
3,12630815,https://omabrowser.org/api/protein/12630815/,MOUSE00344,TSYL1_MOUSE,432fce4f9c5dc63367459555ce43cdeb,379,"{'code': 'MOUSE', 'taxon_id': 10090, 'species'...",916750,HOG:E0726718.2b,10,"{'start': 34158277, 'end': 34159416, 'strand': 1}",True,HOG:E0726718
4,12631693,https://omabrowser.org/api/protein/12631693/,MOUSE01222,Q3TZU5,066e30d0845bcbd6ec0e3a0ab958043c,78,"{'code': 'MOUSE', 'taxon_id': 10090, 'species'...",1388946,HOG:E0726796.3a,10,"{'start': 57604194, 'end': 57612453, 'strand': 1}",True,HOG:E0726796
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1250,12631722,https://omabrowser.org/api/protein/12631722/,MOUSE01251,M0QWR4,5c449fd2339d6ea912b6b5711bb7367f,261,"{'code': 'MOUSE', 'taxon_id': 10090, 'species'...",0,HOG:E0801468.10pxg.6912a.5506a.4064d.2831b.1712g,10,"{'start': 104005740, 'end': 104006525, 'strand...",True,HOG:E0801468
1161,12631633,https://omabrowser.org/api/protein/12631633/,MOUSE01162,H17B6_MOUSE,34e1a26e93d13a13c239fb23fbd7f0e9,317,"{'code': 'MOUSE', 'taxon_id': 10090, 'species'...",827595,HOG:E0792933.6aw.330f,10,"{'start': 127827117, 'end': 127833879, 'strand...",True,HOG:E0792933
185,12630657,https://omabrowser.org/api/protein/12630657/,MOUSE00186,A0A1W2P7H4,04434d2091f471ddc782d2174c9e899e,802,"{'code': 'MOUSE', 'taxon_id': 10090, 'species'...",1077919,HOG:E0802810.8a,10,"{'start': 118182176, 'end': 118184584, 'strand...",True,HOG:E0802810
1050,12631522,https://omabrowser.org/api/protein/12631522/,MOUSE01051,ENSMUSG00000112856.2,cc601904ffa422af685229b1058f94d5,289,"{'code': 'MOUSE', 'taxon_id': 10090, 'species'...",0,HOG:E0797296.1d,10,"{'start': 100186973, 'end': 100187842, 'strand...",True,HOG:E0797296
